![](https://i.imgur.com/ztInM2d.png)

In [ ]:
!pip install langchain==1.2.15 langchain-groq==1.1.2 langgraph==1.1.8 langchain-mcp-adapters==0.2.2 mcp==1.27.0 --quiet

## Finance MCP Server

This section describes a **FastMCP** server called `FinanceServer`, configured to run on **port `8010`**. The server is started using:

```python
mcp.run(transport="streamable-http")
````

### Overview

The server is initialized via the `FastMCP(...)` constructor, which includes:

* A short instruction string describing its purpose
* An explicit port configuration (`8010`)

### Registered Tools

Two tools are exposed using the `@mcp.tool()` decorator:

#### 1. `generate_invoice(customer_id: str) -> dict`

Generates a mock invoice for a given customer.

**Example response:**

```json
{
  "invoice_id": "INV-1001",
  "customer_id": "<id>",
  "amount": 2500,
  "status": "generated"
}
```

#### 2. `get_budget_summary(department: str) -> dict`

Provides a simple budget overview for a specified department.

**Example response:**

```json
{
  "department": "<dept>",
  "budget": 100000,
  "spent": 65432,
  "remaining": 34568
}
```


In [ ]:
%%writefile finance_mcp_server.py
from mcp.server.fastmcp import FastMCP

# Server with instruction
mcp = FastMCP(
    name="FinanceServer",
    instructions = """
          This server provides finance related tools
          - Call generate_invoice(customer_id) to generate a new invoice for a customer.
          - Call get_budget_summary(department) to get a summary of a department's budget.
    """,
    port=8010
)
@mcp@tool()
def generate_invoice(customer_id: str) -> dict:
    return {


    }

In [ ]:
%%writefile finance_mcp_server.py

from mcp.server.fastmcp import FastMCP

# Server with instructions
mcp = FastMCP(
    name="FinanceServer",
    instructions="""
        This server provides finance-related tools.
        - Call generate_invoice(customer_id) to generate a new invoice for a customer.
        - Call get_budget_summary(department) to retrieve the budget overview of any department.
    """,
    port=8010
)

@mcp.tool()
def generate_invoice(customer_id: str) -> dict:
    return {
        "invoice_id": "INV-1001",
        "customer_id": customer_id,
        "amount": 2500,
        "status": "generated"
    }

@mcp.tool()
def get_budget_summary(department: str) -> dict:
    return {
        "department": department,
        "budget": 100000,
        "spent": 65432,
        "remaining": 34568
    }

if __name__ == "__main__":
    print("Starting Finance MCP Server...")
    mcp.run(transport="streamable-http")


### Deploying the MCP Server for the Finance Department
The Finance server is launched as a background process in the notebook server:
- Starts FinanceServer on ` http://localhost:8010/mcp` using the `streamable-http` transport.
- Logs are written to `finance_server_output.log`.

In [ ]:
!nohup python /content/finance_mcp_server.py > finance_server_output.log 2>&1 &

## Building a MCP Server for the HR Department

This section defines a **FastMCP** server named `HRServer` that runs on **port `8011`** and is started with **`mcp.run(transport="streamable-http")`**.

It registers two tools via `@mcp.tool()`:

- `get_employee_details(employee_id: str) -> dict`  
  Returns a stubbed employee profile, e.g.  
  `{"employee_id": "<id>", "name": "Alice Johnson", "role": "Software Engineer", "department": "Tech"}`

- `check_leave_balance(employee_id: str) -> dict`  
  Returns leave information, e.g.  
  `{"employee_id": "<id>", "leave_balance": 12}`

Like the Finance server, `FastMCP(...)` includes instructions and the explicit port.


In [ ]:
%%writefile hr_mcp_server.py

from mcp.server.fastmcp import FastMCP

# Server with instructions
mcp = FastMCP(
    name="HRServer",
    instructions="""
        This server provides human resources tools.
        - Call get_employee_details(employee_id) to fetch an employee's information.
        - Call check_leave_balance(employee_id) to view the available leave balance for an employee.
    """,
    port=8011
)

@mcp.tool()
def get_employee_details(employee_id: str) -> dict:
    return {
        "employee_id": employee_id,
        "name": "Alice Johnson",
        "role": "Software Engineer",
        "department": "Tech"
    }

@mcp.tool()
def check_leave_balance(employee_id: str) -> dict:
    return {
        "employee_id": employee_id,
        "leave_balance": 12
    }

if __name__ == "__main__":
    print("Starting HR MCP Server...")
    mcp.run(transport="streamable-http")

## Deploying the MCP Server for the HR Department

The HR server is launched as a background process in the notebook server:
- Starts HRServer on ` http://localhost:8011/mcp` using the `streamable-http` transport.
- Logs are written to `hr_server_output.log`.

In [ ]:
!nohup python /content/hr_mcp_server.py > hr_server_output.log 2>&1 &

## MCP Client Agent Setup

This section defines a client-side agent that connects to multiple MCP servers and enables unified tool usage across them.

### MCP Client Configuration

A `MultiServerMCPClient` is initialized with connections to both the Finance and HR servers:

```python
client = MultiServerMCPClient({
    "finance": {"url": "http://localhost:8010/mcp", "transport": "streamable_http"},
    "hr":      {"url": "http://localhost:8011/mcp", "transport": "streamable_http"}
})
````

This allows the client to communicate with multiple MCP servers through a single interface.

---

### Language Model Setup
```python
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY)
```
---

### Tool Discovery

The client dynamically fetches all available tools from connected servers:

```python
tools = await client.get_tools()
print("Discovered tools:", [tool.name for tool in tools])
```

This will include tools from both:

* `FinanceServer`
* `HRServer`

---

### Agent Creation

A ReAct-style agent is created using the discovered tools:

```python
agent = create_react_agent(model=llm, tools=tools)
```

---

## Outcome

This setup results in a **single intelligent agent** that can:

* Discover tools from multiple MCP servers
* Invoke finance-related tools (e.g., invoice generation, budget summary)
* Invoke HR-related tools (e.g., employee data, leave management)
* Seamlessly reason across domains using one unified interface

In short, the agent acts as a centralized brain that orchestrates capabilities from both **FinanceServer** and **HRServer** via MCP.


In [ ]:
%%writefile client_agent.py

from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
import asyncio
from dotenv import load_dotenv

load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY)

async def main():
    client = MultiServerMCPClient({
        "finance": {
            "url": "http://localhost:8010/mcp",
            "transport": "streamable_http"
        },
        "hr": {
            "url": "http://localhost:8011/mcp",
            "transport": "streamable_http"
        }
    })

    tools = await client.get_tools()
    print("Discovered tools:", [tool.name for tool in tools])

    agent = create_react_agent(model=llm, tools=tools)

    # Sample queries
    print('\nUsing capabilities from Finance - Invoice Generation:')
    result_1 = await agent.ainvoke({"messages": [{"role": "user", "content": "Can you generate an invoice for customer 123?"}]})
    print(result_1['messages'][-1].content)

    print('\nUsing capabilities from HR - Leave Balance:')
    result_2 = await agent.ainvoke({"messages": [{"role": "user", "content": "Show me the leave balance for employee 456"}]})
    print(result_2['messages'][-1].content)

    print('\nUsing capabilities from Finance - Budget Planning:')
    result_3 = await agent.ainvoke({"messages": [{"role": "user", "content": "What is the tech department’s budget summary?"}]})
    print(result_3['messages'][-1].content)

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
!nohup python /content/client_agent.py > client_agent.log 2>&1 &